# STEP 5 — Temporal Holdout Validation  
======================================  
Everything in Steps 2-4 is DESCRIPTIVE: it tells you what patterns exist in  
the data you already have. It does NOT tell you whether those patterns  
would have correctly predicted something you didn't yet know. This step  
closes that gap.  
  
Method: split the data by TIME, not randomly.  
  - TRAIN window: earlier portion of the date range — mine rules from this  
    period only, exactly as Steps 2-4 already do.  
  - TEST window: later portion — for each surviving rule, check customers  
    who had the antecedent basket in the TRAIN window but did NOT yet have  
    the consequent. Did they go on to buy the consequent during the TEST  
    window? This is the actual "would this recommendation have worked"  
    question, and nothing before this step answers it.  
  
This is a real, if imperfect, predictive validation — imperfect because we  
can't know whether a customer bought the consequent BECAUSE of an  
actual recommendation (none was ever shown to them; we're only checking  
whether the pattern continues to hold), only whether the pattern  
generalizes forward in time rather than being a one-off artifact of the  
training window. State this limitation plainly in the write-up — it is a  
genuine, known limitation of retrospective validation.  
  
Input:  
  - clean_transactions_all_drugs.parquet (Step 1)  
  - final_recommendations_genuine_opportunity.csv (Step 4, whole-dataset)  
  - category_rules_final_genuine_opportunity.csv (Step 4, category-wise) — optional  
  
Output: holdout_validation_results.parquet / .csv — precision per rule:  
  what fraction of the "genuine opportunity" audience actually bought the  
  recommended product in the following period.


No EDW authentication needed — runs entirely off Step 1/4's saved output files.

In [1]:
import pandas as pd
import numpy as np
import ast

### CONFIG

In [2]:
# Split point: adjust to match your actual DATE_RANGE from Step 1. This
# should leave a meaningful amount of data on BOTH sides — too early a
# split starves the training rules of data, too late leaves no room to
# observe outcomes.
TRAIN_END_DATE = "2025-06-01"

### PART A — Load transactions and split by time

In [3]:
transactions = pd.read_parquet("clean_transactions_all_drugs.parquet")
transactions["dispense_date_created"] = pd.to_datetime(transactions["dispense_date_created"])

train_txns = transactions[transactions["dispense_date_created"] < TRAIN_END_DATE]
test_txns = transactions[transactions["dispense_date_created"] >= TRAIN_END_DATE]

print(f"[A] Train window: {len(train_txns):,} rows "
      f"({train_txns['dispense_date_created'].min().date()} to "
      f"{train_txns['dispense_date_created'].max().date()})")
print(f"[A] Test window:  {len(test_txns):,} rows "
      f"({test_txns['dispense_date_created'].min().date()} to "
      f"{test_txns['dispense_date_created'].max().date()})")

if len(train_txns) == 0 or len(test_txns) == 0:
    raise SystemExit(
        "TRAIN_END_DATE splits all data to one side — adjust it to fall "
        "inside your actual DATE_RANGE from Step 1."
    )

[A] Train window: 42,688,190 rows (2024-01-01 to 2025-05-31)
[A] Test window:  24,435,810 rows (2025-06-01 to 2025-12-31)


### PART B — Load the surviving rules from Step 4 (whole-dataset + category)

In [4]:
def _parse_tuple_col(series):
    """CSV round-trips tuples as string reprs like "('25847.0',)" — parse
    them back into real tuples of floats for set operations below."""
    def parse_one(x):
        if isinstance(x, tuple):
            return x
        parsed = ast.literal_eval(x)
        return tuple(float(v) for v in parsed)
    return series.apply(parse_one)

rules_to_test = []

try:
    whole_rules = pd.read_csv("final_recommendations_genuine_opportunity.csv")
    whole_rules["antecedents"] = _parse_tuple_col(whole_rules["antecedents"])
    whole_rules["consequents"] = _parse_tuple_col(whole_rules["consequents"])
    whole_rules["source"] = "whole_dataset"
    rules_to_test.append(whole_rules)
    print(f"[B] Loaded {len(whole_rules)} whole-dataset genuine-opportunity rules")
except FileNotFoundError:
    print("[B] final_recommendations_genuine_opportunity.csv not found — skipping")

try:
    category_rules = pd.read_csv("category_rules_final_genuine_opportunity.csv")
    category_rules["antecedents"] = _parse_tuple_col(category_rules["antecedents"])
    category_rules["consequents"] = _parse_tuple_col(category_rules["consequents"])
    category_rules["source"] = "category_wise"
    rules_to_test.append(category_rules)
    print(f"[B] Loaded {len(category_rules)} category-wise genuine-opportunity rules")
except FileNotFoundError:
    print("[B] category_rules_final_genuine_opportunity.csv not found — skipping "
          "(run Step 4 Part F first if you want category-wise rules included)")

if not rules_to_test:
    raise SystemExit(
        "No rule files found. Run Step 4 (04_safety_validation.ipynb) first "
        "to produce final_recommendations_genuine_opportunity.csv."
    )

rules_df = pd.concat(rules_to_test, ignore_index=True)
print(f"[B] {len(rules_df)} total rules to validate")

[B] Loaded 112 whole-dataset genuine-opportunity rules
[B] category_rules_final_genuine_opportunity.csv not found — skipping (run Step 4 Part F first if you want category-wise rules included)
[B] 112 total rules to validate


### PART C — Rebuild TRAIN-window baskets (per customer, whole train period — using the whole period rather than weekly here since we're checking "did they ever have this antecedent during training", not re-deriving support/confidence)

In [5]:
train_customer_products = (
    train_txns.groupby("CustomerKey")["ProductKey"].apply(lambda s: frozenset(s.unique()))
)
test_customer_products = (
    test_txns.groupby("CustomerKey")["ProductKey"].apply(lambda s: frozenset(s.unique()))
)

print(f"[C] {len(train_customer_products):,} customers active in train window")
print(f"[C] {len(test_customer_products):,} customers active in test window")

[C] 1,450,049 customers active in train window
[C] 1,543,289 customers active in test window


### PART D — For each rule, check holdout precision

In [6]:
def holdout_check(antecedent, consequent, train_products, test_products):
    antecedent_set = set(antecedent)
    consequent_set = set(consequent)

    # Customers who had the FULL antecedent in TRAIN but NOT the consequent yet
    eligible_customers = [
        cust for cust, prods in train_products.items()
        if antecedent_set.issubset(prods) and not consequent_set.issubset(prods)
    ]
    if len(eligible_customers) == 0:
        return pd.Series({
            "n_eligible_customers": 0, "n_bought_in_test": 0, "holdout_precision": np.nan,
        })

    n_bought = sum(
        1 for cust in eligible_customers
        if cust in test_products and consequent_set.issubset(test_products[cust])
    )
    return pd.Series({
        "n_eligible_customers": len(eligible_customers),
        "n_bought_in_test": n_bought,
        "holdout_precision": n_bought / len(eligible_customers),
    })

print("[D] Running holdout check per rule "
      f"({len(rules_df)} rules x {len(train_customer_products):,} train customers)")

holdout_results = rules_df.apply(
    lambda r: holdout_check(r["antecedents"], r["consequents"], train_customer_products, test_customer_products),
    axis=1,
)
rules_df = pd.concat([rules_df, holdout_results], axis=1)

[D] Running holdout check per rule (112 rules x 1,450,049 train customers)


### PART E — Baseline comparison: what fraction of ALL eligible customers (regardless of antecedent) buy the consequent in the test window? This tells you whether the rule beats random chance, not just whether SOME customers bought the product.

In [7]:
def baseline_rate(consequent, train_products, test_products):
    consequent_set = set(consequent)
    # All customers active in train who don't yet have the consequent
    eligible = [
        cust for cust, prods in train_products.items()
        if not consequent_set.issubset(prods)
    ]
    if len(eligible) == 0:
        return np.nan
    n_bought = sum(
        1 for cust in eligible
        if cust in test_products and consequent_set.issubset(test_products[cust])
    )
    return n_bought / len(eligible)

# Cache baseline rate per distinct consequent (expensive to recompute per rule)
baseline_cache = {}
def get_baseline(consequent):
    if consequent not in baseline_cache:
        baseline_cache[consequent] = baseline_rate(consequent, train_customer_products, test_customer_products)
    return baseline_cache[consequent]

print("[E] Computing baseline (random chance) purchase rate per consequent product")
rules_df["baseline_rate"] = rules_df["consequents"].apply(get_baseline)
rules_df["lift_over_baseline"] = rules_df["holdout_precision"] / rules_df["baseline_rate"]

[E] Computing baseline (random chance) purchase rate per consequent product


### Save and summarize

In [8]:
rules_df["antecedents"] = rules_df["antecedents"].apply(lambda t: tuple(sorted(t)))
rules_df["consequents"] = rules_df["consequents"].apply(lambda t: tuple(sorted(t)))
rules_df.to_parquet("holdout_validation_results.parquet", index=False)
rules_df.to_csv("holdout_validation_results.csv", index=False)

print(f"\nSaved {len(rules_df)} holdout-validated rules -> "
      "holdout_validation_results.parquet / .csv")

print("\n=== Table: Holdout validation results ===")
display_cols = [
    "source", "antecedent_names", "consequent_names",
    "n_eligible_customers", "n_bought_in_test", "holdout_precision",
    "baseline_rate", "lift_over_baseline",
]
print(
    rules_df[rules_df["n_eligible_customers"] >= 10]  # ignore rules with too few eligible customers to be meaningful
    .sort_values("lift_over_baseline", ascending=False)
    [display_cols]
    .to_string(index=False)
)

print(
    "\nInterpretation: holdout_precision is the fraction of eligible customers "
    "(had the antecedent, didn't yet have the consequent) who bought the "
    "consequent in the LATER, held-out period. baseline_rate is what fraction "
    "of ALL customers (regardless of antecedent) did the same. "
    "lift_over_baseline > 1 means the rule predicts meaningfully better than "
    "chance; close to 1 means the antecedent adds little real predictive value "
    "beyond the product's general popularity."
)


Saved 112 holdout-validated rules -> holdout_validation_results.parquet / .csv

=== Table: Holdout validation results ===
       source                                                                        antecedent_names                                              consequent_names  n_eligible_customers  n_bought_in_test  holdout_precision  baseline_rate  lift_over_baseline
whole_dataset                                          ONE-ALPHA capsules 250nanograms [NEON HC] [30]                            Renavit tablets [STAN PHARM] [100]                1463.0              33.0           0.022556       0.000073          308.428642
whole_dataset                                RANOLAZINE prolonged release tablet 500mg [KRKA UK] [60]                  ASPIRIN dispersible tablet 75mg [ASPAR] [28]                 337.0               2.0           0.005935       0.000070           85.216012
whole_dataset                                   RANOLAZINE prolonged release tablet 500mg [TEVA] [60]  

In [11]:
n_min = 30  # same statistical-reliability logic used earlier in Step 2's
             # sweep (min_count threshold) — a group this small can't
             # reliably distinguish a real non-replication from noise

real_failures = rules_df[
    (rules_df['lift_over_baseline'] <= 1) & (rules_df['n_eligible_customers'] >= n_min)
]
too_small_to_judge = rules_df[
    (rules_df['lift_over_baseline'] <= 1) & (rules_df['n_eligible_customers'] < n_min)
]
generalized = rules_df[rules_df['lift_over_baseline'] > 1]

print(f"Total rules: {len(rules_df)}")
print(f"Generalized (lift_over_baseline > 1): {len(generalized)}")
print(f"Real non-replications (lift<=1, n>={n_min}): {len(real_failures)}")
print(f"Too small to judge (lift<=1, n<{n_min}): {len(too_small_to_judge)}")

print("\n=== Real non-replications (large enough sample to trust the result) ===")
print(real_failures[["antecedent_names", "consequent_names", "n_eligible_customers",
                       "n_bought_in_test", "holdout_precision", "baseline_rate",
                       "lift_over_baseline"]].sort_values("n_eligible_customers", ascending=False).to_string(index=False))

real_failures.to_csv("holdout_real_non_replications.csv", index=False)
generalized.to_csv("holdout_generalized_rules.csv", index=False)
print("\nSaved: holdout_real_non_replications.csv, holdout_generalized_rules.csv")

Total rules: 112
Generalized (lift_over_baseline > 1): 90
Real non-replications (lift<=1, n>=30): 14
Too small to judge (lift<=1, n<30): 3

=== Real non-replications (large enough sample to trust the result) ===
                                                                       antecedent_names                                              consequent_names  n_eligible_customers  n_bought_in_test  holdout_precision  baseline_rate  lift_over_baseline
MOUNJARO KWIKPEN solution for injection 2.4ml pre-filled pen 10mg/0.6ml [ELI LILLY] [1]                ASPIRIN dispersible tablet 75mg [ACTAVIS] [28]               16998.0              15.0           0.000882       0.003026            0.291630
                                                  EZETIMIBE tablets 10mg [Liconsa] [28]                  ASPIRIN dispersible tablet 75mg [ASPAR] [28]                8039.0               0.0           0.000000       0.000070            0.000000
                                  OMEPRAZOLE gastro-resi

In [12]:
transactions = pd.read_parquet("clean_transactions_all_drugs.parquet")
transactions["dispense_date_created"] = pd.to_datetime(transactions["dispense_date_created"])
aspar_mask = transactions["DrugName"].str.contains("ASPAR", case=False, na=False)
print(transactions[aspar_mask].groupby(transactions["dispense_date_created"].dt.to_period("M")).size())

dispense_date_created
2024-01      488
2024-02      390
2024-03     1856
2024-04      572
2024-05     7065
2024-06    23322
2024-07    28182
2024-08    26781
2024-09    27604
2024-10    29025
2024-11    27039
2024-12    20154
2025-01     1846
2025-02      379
2025-03      365
2025-04      397
2025-05      278
2025-06      688
2025-07      680
2025-08      425
2025-09      585
2025-10      512
2025-11      357
2025-12      447
Freq: M, dtype: int64


In [9]:
print(f"Rules with lift_over_baseline > 1: {(rules_df['lift_over_baseline'] > 1).sum()} / {len(rules_df)}")
print(f"Rules with lift_over_baseline <= 1 (did NOT generalize):")
print(rules_df[rules_df['lift_over_baseline'] <= 1][['antecedent_names','consequent_names','lift_over_baseline','n_eligible_customers']].to_string(index=False))

Rules with lift_over_baseline > 1: 90 / 112
Rules with lift_over_baseline <= 1 (did NOT generalize):
                                                                       antecedent_names                                              consequent_names  lift_over_baseline  n_eligible_customers
MOUNJARO KWIKPEN solution for injection 2.4ml pre-filled pen 10mg/0.6ml [ELI LILLY] [1]                ASPIRIN dispersible tablet 75mg [ACTAVIS] [28]            0.291630               16998.0
                                                 DAPAGLIFLOZIN tablets 10mg [TEVA] [28]                ASPIRIN dispersible tablet 75mg [ACTAVIS] [28]            0.000000                   1.0
                                  OMEPRAZOLE gastro-resistant capsules 10mg [TEVA] [28]                ASPIRIN dispersible tablet 75mg [ACTAVIS] [28]            0.912916                6516.0
                                               ROSUVASTATIN tablets 20mg [KRKA UK] [28]                  ASPIRIN dispersible tablet

In [10]:
print(rules_df['lift_over_baseline'].describe())

count    107.000000
mean       7.040206
std       31.200494
min        0.000000
25%        1.341603
50%        2.253608
75%        3.395890
max      308.428642
Name: lift_over_baseline, dtype: float64
